# DeepSentinel: End-to-End Training & 10,000-Clip Benchmark Evaluation

This notebook runs the **Free Google Colab (T4 GPU)** training pipeline and evaluates out-of-domain performance on **FakeAVCeleb**.

### **Pipeline Stages**:
1. **Stage 1 (Head Pre-training)**: Pre-trains the 299-D multi-scale fusion head, SE-Attention block, and GELU activations (~60 seconds).
2. **Stage 2 (Backbone Fine-tuning)**: Fine-tunes Wav2Vec2/ViT backbones with FP16, `pos_weight = 1.3835`, LayerNorm, and differential learning rates (~45 minutes).
3. **FakeAVCeleb Benchmark**: Evaluates on 10,000 cached FakeAVCeleb clips with Youden's J Statistic sweep (~30 seconds).

In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 1: Pull latest commit with complete stabilized pipeline
!rm -rf /content/thesis
!git clone -b feat/training-turnover-prep https://github.com/gjvlio/emotion-based-multimodal-deepfake-detector.git /content/thesis
%cd /content/thesis
!git reset --hard db0c0d56f51123d819853f428b481ed03f14d63c
!pip install -q transformers scikit-learn tensorboard timm pandas openai-whisper opencv-python-headless

In [ ]:
# Cell 2: Run Stage 1 (50 Full Epochs in ~2 minutes)
!python scripts/colab_stage1.py


In [ ]:
# Step 4: Run Stage 2 Backbone Fine-Tuning & 10,000-Clip Benchmark Evaluation (~45 minutes)
!python scripts/colab_stage2.py

In [ ]:
# Cell 4: Evaluate final fine-tuned model on 10,000 FakeAVCeleb clips
!python scripts/evaluate_fakeavceleb.py --checkpoint checkpoints/full/bottleneck_mode/best_phase2_bottleneck.pt --classifier_mode bottleneck --n_real 500 --n_fake 9500 --save_csv benchmark_results_bottleneck.csv
